In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import gcamp_peak_utils as gp

In [2]:
def avg_event_rate_by_speed_bins(df, idx_by_bin, speed_col='linearSpeedcmPerSecond'):
    """
    Given:
      • df            : DataFrame with one column per cell (instantaneous rates)
                        plus a `speed_col`.
      • idx_by_bin    : dict mapping (low, high) tuples → list of row-indices
                        where speed ∈ [low, high) (or >= low if high is inf).
      • speed_col     : name of the speed column in df (will be dropped).
    Returns:
      • avg_df        : DataFrame indexed by your (low, high) bins,
                        columns are the cell-names, entries are the mean
                        event-rate of that cell over all frames in that bin.
    """
    # 1) Identify all “cell” columns (everything except the speed column)
    event_cols = [c for c in df.columns if c != speed_col]

    # 2) Make a DataFrame to hold means; use a MultiIndex of your bin tuples
    bin_index = pd.MultiIndex.from_tuples(idx_by_bin.keys(), names=['low','high'])
    avg_df = pd.DataFrame(index=bin_index, columns=event_cols, dtype=float)

    # 3) For each bin, pull those rows and take the column‐wise mean
    for bin_range, idxs in idx_by_bin.items():
        if len(idxs)>0:
            avg_df.loc[bin_range] = df.loc[idxs, event_cols].mean()
        else:
            avg_df.loc[bin_range] = np.nan

    return avg_df

In [ ]:
base_dir = Path("/projects/b1118/CaliAli_linearTrackData")
analysis_dirs = sorted(base_dir.glob("m*_analysis"))
summary_rows = []
for analysis_dir in analysis_dirs:
    gcamp_path = analysis_dir / "GCAMP_with_velocity.csv"
    out_path = analysis_dir / "all_peak_stats.csv"
    if not gcamp_path.exists():
        print(f"Skipping {analysis_dir.name}: no GCAMP_with_velocity.csv")
        continue

    print(f"Processing {analysis_dir.name}")

    gcamp_df = gp.load_aligned_gcamp_df(gcamp_path)
    cell_cols = [c for c in gcamp_df.columns if c.startswith("cell_")]

    if not cell_cols:
        print(f"  No cell columns found in {gcamp_path.name}")
        continue

    aligned_GCAMP = gcamp_df[cell_cols]

    signalPeaks = gp.detect_peaks_df(
        aligned_GCAMP,
        threshold=2.5,
        smooth_window_samples=3,
        min_peak_prominence=1.5,
        min_peak_distance_samples=20,
        min_peak_width_samples=2,
        return_to_baseline_tol=0.4,
        return_to_baseline_window_samples=5,
    )

    all_stats_df = gp.compute_peak_stats_for_all_cells_from_gcamp_df(
        gcamp_df,
        signalPeaks,
        tol=0.25,
        window_len_onset=3,
        window_len_offset=3,
        plt_region=10,
        behavior_columns=["X_coor", "Y_coor", "Velocity_spatial_filtered"],
        output_csv_path=out_path,
    )

    summary_rows.append({
        "analysis_dir": analysis_dir.name,
        "gcamp_file": str(gcamp_path),
        "output_csv": str(out_path),
        "n_cells": len(cell_cols),
        "n_events": len(all_stats_df),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

Processing m311_analysis


In [ ]:
base_dir = Path("/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays")
analysis_dirs = sorted(base_dir.glob("m*_analysis"))

velocity_bins = [(0, 0.5), (0.5, 2), (2, 5), (5, 10), (10, np.inf)]

summary_rows = []
mismatch_rows = []

for analysis_dir in analysis_dirs:
    gcamp_path = analysis_dir / "GCAMP_with_velocity.csv"
    rate_path = analysis_dir / "instantaneousEventRate.csv"
    rate_onsets_path = analysis_dir / "instantaneousEventRateOnsets.csv"
    avg_out_path = analysis_dir / "avg_event_rate_by_speed_bins.csv"
    avg_onsets_out_path = analysis_dir / "avg_event_rate_onsets_by_speed_bins.csv"

    if not (gcamp_path.exists() and rate_path.exists() and rate_onsets_path.exists()):
        summary_rows.append({
            "analysis_dir": analysis_dir.name,
            "status": "missing_input_csv"
        })
        continue

    try:
        print(f"Processing {analysis_dir.name}")

        GCAMP_with_velocity = pd.read_csv(gcamp_path, index_col=0).reset_index(drop=True)
        instantaneousEventRate = pd.read_csv(rate_path, index_col=0).reset_index(drop=True)
        instantaneousEventRateOnsets = pd.read_csv(rate_onsets_path, index_col=0).reset_index(drop=True)

        len_gcamp = len(GCAMP_with_velocity)
        len_rate = len(instantaneousEventRate)
        len_onsets = len(instantaneousEventRateOnsets)

        if len_rate != len_gcamp or len_onsets != len_gcamp:
            mismatch_rows.append({
                "analysis_dir": analysis_dir.name,
                "len_gcamp": len_gcamp,
                "len_rate": len_rate,
                "len_onsets": len_onsets,
            })

        velocity = GCAMP_with_velocity["Velocity_spatial_filtered"]

        instantaneousEventRate["Velocity_spatial_filtered"] = velocity.iloc[:len_rate].reset_index(drop=True)
        instantaneousEventRateOnsets["Velocity_spatial_filtered"] = velocity.iloc[:len_onsets].reset_index(drop=True)

        velocity_bin_ilocs_rate = {
            (lo, hi): np.flatnonzero(
                (
                    (instantaneousEventRate["Velocity_spatial_filtered"] >= lo) &
                    ((instantaneousEventRate["Velocity_spatial_filtered"] < hi) if np.isfinite(hi) else True)
                ).to_numpy()
            ).tolist()
            for lo, hi in velocity_bins
        }

        velocity_bin_ilocs_onsets = {
            (lo, hi): np.flatnonzero(
                (
                    (instantaneousEventRateOnsets["Velocity_spatial_filtered"] >= lo) &
                    ((instantaneousEventRateOnsets["Velocity_spatial_filtered"] < hi) if np.isfinite(hi) else True)
                ).to_numpy()
            ).tolist()
            for lo, hi in velocity_bins
        }

        avg_rates = avg_event_rate_by_speed_bins(
            instantaneousEventRate,
            velocity_bin_ilocs_rate,
            speed_col="Velocity_spatial_filtered"
        )

        avg_rates_onsets = avg_event_rate_by_speed_bins(
            instantaneousEventRateOnsets,
            velocity_bin_ilocs_onsets,
            speed_col="Velocity_spatial_filtered"
        )

        avg_rates.to_csv(avg_out_path)
        avg_rates_onsets.to_csv(avg_onsets_out_path)

        summary_rows.append({
            "analysis_dir": analysis_dir.name,
            "status": "ok",
            "len_gcamp": len_gcamp,
            "len_rate": len_rate,
            "len_onsets": len_onsets,
            "avg_rates_csv": str(avg_out_path),
            "avg_rates_onsets_csv": str(avg_onsets_out_path),
        })

    except Exception as e:
        summary_rows.append({
            "analysis_dir": analysis_dir.name,
            "status": "error",
            "error": str(e),
        })
        continue

summary_df = pd.DataFrame(summary_rows)
mismatch_df = pd.DataFrame(mismatch_rows)

summary_df



Processing m311_analysis
Processing m326_analysis


In [ ]:
summary_df

In [ ]:
## mice are 
# WT: [992, 994, 989, 757, 326, 311]
# KO: [752, 328, 388]

In [ ]:
base_dir = Path("/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays")
analysis_dirs = sorted(base_dir.glob("m*_analysis"))

all_rates = []
all_onsets = []
missing = []

for analysis_dir in analysis_dirs:
    rates_path = analysis_dir / "avg_event_rate_by_speed_bins.csv"
    onsets_path = analysis_dir / "avg_event_rate_onsets_by_speed_bins.csv"

    if not rates_path.exists() or not onsets_path.exists():
        missing.append(analysis_dir.name)
        continue

    mouse_name = analysis_dir.name.replace("_analysis", "")

    # read multiindex rows: low, high
    rates_df = pd.read_csv(rates_path, index_col=[0, 1])
    onsets_df = pd.read_csv(onsets_path, index_col=[0, 1])

    # long format
    rates_long = (
        rates_df
        .reset_index()
        .rename(columns={"low": "speed_bin_low", "high": "speed_bin_high"})
        .melt(
            id_vars=["speed_bin_low", "speed_bin_high"],
            var_name="cell",
            value_name="avg_rate"
        )
    )
    rates_long["mouse"] = mouse_name

    onsets_long = (
        onsets_df
        .reset_index()
        .rename(columns={"low": "speed_bin_low", "high": "speed_bin_high"})
        .melt(
            id_vars=["speed_bin_low", "speed_bin_high"],
            var_name="cell",
            value_name="avg_rate_onsets"
        )
    )
    onsets_long["mouse"] = mouse_name

    all_rates.append(rates_long)
    all_onsets.append(onsets_long)

all_rates_df = pd.concat(all_rates, ignore_index=True) if all_rates else pd.DataFrame()
all_onsets_df = pd.concat(all_onsets, ignore_index=True) if all_onsets else pd.DataFrame()

missing


In [ ]:
## mice are 
# WT: [992, 994, 989, 757, 326, 311]
# KO: [752, 328, 388]

In [ ]:
m311_onsets_avg_across_cells = (
    all_onsets_df.loc[all_onsets_df["mouse"] == "m311"]
    .groupby(["speed_bin_low", "speed_bin_high"], dropna=False)["avg_rate_onsets"]
    .mean()
    .reset_index(name="mean_avg_rate_onsets_across_cells")
)

m311_onsets_avg_across_cells

In [ ]:
all_onsets_df

In [ ]:
## load everything from the *saved events .csv file 
def get_event_rate_same_length(df, samplingRate, pad_value=0):
    """
    Forward-looking 1-second window sums, same length as input.
    """
    arr = df.to_numpy(dtype=int)
    kernel = np.ones(samplingRate, dtype=int)

    pad = samplingRate - 1
    arr_pad = np.pad(arr, ((0, pad), (0, 0)), mode="constant", constant_values=pad_value)

    rates = np.apply_along_axis(
        lambda col: np.convolve(col, kernel, mode="valid"),
        axis=0,
        arr=arr_pad
    )
    return pd.DataFrame(rates, index=df.index, columns=df.columns)


base_dir = Path("/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_LF_CaImaging/LinearTrackDataAlignedAcrossDays")
analysis_dirs = sorted(base_dir.glob("m*_analysis"))

samplingRate = 20
event_index_col = "onset_idx"   # use "peak_idx" instead if you want local-max timing rather than onset timing

summary_rows = []

for analysis_dir in analysis_dirs:
    gcamp_path = analysis_dir / "GCAMP_with_velocity.csv"
    stats_path = analysis_dir / "all_peak_stats.csv"
    out_path = analysis_dir / "instantaneousEventRateOnsets_LocalMax.csv"

    if not (gcamp_path.exists() and stats_path.exists()):
        summary_rows.append({
            "analysis_dir": analysis_dir.name,
            "status": "missing_input_csv"
        })
        continue

    print(f"Processing {analysis_dir.name}")

    try:
        gcamp_df = pd.read_csv(gcamp_path, index_col=0).reset_index(drop=True)
        stats_df = pd.read_csv(stats_path)

        cell_cols = [c for c in gcamp_df.columns if c.startswith("cell_")]
        n_rows = len(gcamp_df)

        # binary matrix of conservative event onsets
        localmax_onsets = pd.DataFrame(0, index=np.arange(n_rows), columns=cell_cols, dtype=int)

        if not stats_df.empty:
            required_cols = {"cell", event_index_col}
            missing_cols = required_cols - set(stats_df.columns)
            if missing_cols:
                raise KeyError(f"{stats_path.name} missing required columns: {missing_cols}")

            events_df = (
                stats_df.loc[:, ["cell", event_index_col]]
                .dropna()
                .copy()
            )
            events_df[event_index_col] = events_df[event_index_col].astype(int)
            events_df = events_df[events_df["cell"].isin(cell_cols)]
            events_df = events_df[
                (events_df[event_index_col] >= 0) &
                (events_df[event_index_col] < n_rows)
            ]

            for cell_name, idxs in events_df.groupby("cell")[event_index_col]:
                localmax_onsets.loc[np.unique(idxs.to_numpy()), cell_name] = 1

        instantaneousEventRateOnsets_LocalMax = (
            get_event_rate_same_length(localmax_onsets, samplingRate=samplingRate)
            .reset_index(drop=True)
        )

        instantaneousEventRateOnsets_LocalMax.to_csv(out_path, index=True)

        summary_rows.append({
            "analysis_dir": analysis_dir.name,
            "status": "ok",
            "n_rows": n_rows,
            "n_cells": len(cell_cols),
            "n_events_from_stats": 0 if stats_df.empty else len(stats_df),
            "output_csv": str(out_path),
        })

    except Exception as e:
        summary_rows.append({
            "analysis_dir": analysis_dir.name,
            "status": "error",
            "error": str(e),
        })

summary_df_localmax_onsets = pd.DataFrame(summary_rows)
summary_df_localmax_onsets
